# ABSA tiếng Việt: 7 khía cạnh, 21 nhãn

So sánh TF-IDF + Logistic Regression, Linear SVM và NB-SVM. Hạt giống 2026.
Điểm chính là macro F1 của 14 nhãn khen/chê, theo mã chấm gốc.

Không gọi API/LLM, không hiển thị văn bản test. Dữ liệu bổ sung: **không**.
LLM hỗ trợ viết code và phân tích lỗi trên train; không sinh nhãn test.

Giao thức: khoảng 60% fit, 20% tuning, 20% validation, nhóm theo văn bản chuẩn hóa.
Vectorizer và tỷ số NB chỉ fit trên fit khi đánh giá. Ngưỡng và lựa chọn mô hình
chỉ dùng tuning. Đã xem điểm baseline trước khi thêm NB-SVM; điểm validation
vì thế có yếu tố phản hồi trong quá trình phát triển, không phải holdout xác nhận
hoàn toàn độc lập. Xem HUONG_DAN.md để biết giới hạn và cách chạy.

In [1]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
import pipeline
import score_absa
DATA = Path(os.environ.get('ABSA_DATA_DIR', 'data')).resolve()
OUT = Path('results')
print('Python:', sys.version.split()[0])
print('Seed:', pipeline.SEED)
print('Nguồn dữ liệu commit: 7c052a71010eac9361ea8803b9e8bdd6cfc99be8')

Python: 3.12.14
Seed: 2026
Nguồn dữ liệu commit: 7c052a71010eac9361ea8803b9e8bdd6cfc99be8


## Huấn luyện, chọn ngưỡng, đánh giá và dự đoán

Chạy đầy đủ pipeline: không dùng kết quả được tính sẵn thay cho huấn luyện.
Bootstrap 1.000 lần theo đánh giá, mức tin cậy 95%. Nhãn không có cả thật và
đoán dương được bỏ khi lấy macro F1 theo code giảng viên. Precision không xác
định được bỏ khi tính trung bình. Accuracy chỉ là chỉ số đối chiếu.

In [2]:
with threadpool_limits(limits=1):
    comparison, selected_report = pipeline.run(DATA, OUT)
display(comparison)

Partitions: 2812 938 938


Features: 53533 seconds: 4.31


logistic tune F1: 0.7648 validation F1: 0.7323


svm tune F1: 0.7709 validation F1: 0.7411


nbsvm tune F1: 0.7809 validation F1: 0.7561


Selected by tuning: nbsvm


Số mẫu: 938
ABSA-F1 (điểm xếp hạng) = 0.756   CI95% [0.739, 0.772]
F1 theo cực: khen 0.832 | chê 0.680 | trung tính (phụ) 0.003 | phát hiện khía cạnh (phụ) 0.888
Nhãn yếu nhất: vi_tri_che F1 = 0.569; độ lệch chuẩn F1 14 nhãn = 0.127
Precision TB = 0.768 | Recall TB = 0.750 | Accuracy TB = 0.921 (14 nhãn)
  khen: P 0.829 R 0.835 | chê: P 0.706 R 0.665
kappa TB = 0.703 | MCC TB = 0.704 | Balanced acc TB = 0.848
F1 vi mô: khía cạnh = 0.898 | (khía cạnh, cực) = 0.777 | khớp đúng cả đánh giá = 0.357
Mốc lớp đa số =   nan; cải thiện tương đối =   nan

Khía cạnh      Nhắc F1 | Khen F1     P     R     κ     n | Chê F1      P     R     κ     n | Trung tính F1  n
nhan_vien      0.937  | 0.913 0.906 0.919 0.814   495 | 0.654  0.683 0.627 0.610   110 | 0.000       0
sach_se        0.968  | 0.971 0.962 0.981 0.957   310 | 0.829  0.829 0.829 0.814    76 | 0.000       0
vi_tri         0.912  | 0.894 0.868 0.921 0.838   315 | 0.569  0.564 0.574 0.542    54 | 0.000       2
phong          0.884  | 0.768

File hợp lệ, có thể nộp.


Saved model reload: predictions match exactly.


{
  "python": "3.12.14",
  "os": "Linux-6.18.57-x86_64-with-glibc2.41",
  "cpu": "INTEL(R) XEON(R) PLATINUM 8573C",
  "logical_cpu": 3,
  "ram": "MemTotal:       10207480 kB",
  "gpu": "Not used; CPU only",
  "feature_seconds": 4.311222114999964,
  "full_fit_seconds": 12.487642709999932,
  "test_predict_seconds": 0.5743376219999163,
  "total_seconds": 37.01031709900002,
  "selected": "nbsvm",
  "seed": 2026,
  "train_rows": 4688,
  "test_rows": 732,
  "source_commit": "7c052a71010eac9361ea8803b9e8bdd6cfc99be8",
  "sha256": {
    "train.csv": "665b581d11c5cb59eb5b3e6e730660fdd28567c64392f2e288e375dd082bd95e",
    "test_blind.csv": "a8452bbc921f8a3442d318fc2488ae6f4df87d04c160ddcb5f5b3b8d11473162",
    "submission_demo.csv": "4a58d1c42010996fd648317a23c6da9358779b15d25f8b74a901bdb5e7e2a30e",
    "score_absa.py": "423bb1c5be023b8ce04264f5a72bdf01ba6844a7fa39f87267b0bfe5116de333"
  },
  "submission_sha256": "15f913519d5ec57d1de452709c0766278d889337b3d3e7c7c3b3b2ba53c17ce2"
}


,model,tune_F1,validation_F1,precision,recall,accuracy,kappa,MCC,balanced_accuracy,CI95_low,CI95_high,fit_seconds,validation_predict_seconds
0,logistic,0.764818,0.732328,0.750238,0.724448,0.915398,0.675256,0.678266,0.832934,0.713260,0.748994,4.764524,0.030101
1,svm,0.770913,0.741081,0.745345,0.745889,0.914864,0.683380,0.686319,0.841967,0.723099,0.756982,2.911976,0.022801
2,nbsvm,0.780852,0.756126,0.767699,0.750061,0.920804,0.702699,0.704407,0.847783,0.739355,0.771781,3.774606,0.167651


## Phân bố nhãn và chỉ số theo khía cạnh

In [3]:
display(pd.read_csv(OUT/'label_counts.csv'))
rows = []
for aspect, polarities in selected_report['theo_khia_canh'].items():
    for polarity in ['khen', 'che']:
        rows.append({'aspect':aspect, 'polarity':polarity, **polarities[polarity]})
display(pd.DataFrame(rows))
weak = min(selected_report['theo_khia_canh'], key=lambda a: sum(selected_report['theo_khia_canh'][a][p]['f1'] for p in ['khen','che'])/2)
print('Khía cạnh khó nhất theo F1 trung bình khen/chê:', weak)
print('Nhãn khó nhất:', selected_report['F1_nhan_thap_nhat'])
print('ABSA-F1:', selected_report['ABSA_F1'], 'CI95:', selected_report['ABSA_F1_CI95'])
print('Giới hạn: validation dưới 0,80; chưa có điểm private.')

,label,fit_positive,tune_positive,val_positive
0,nhan_vien_nhac,1786,598,590
1,nhan_vien_khen,1479,495,495
2,nhan_vien_che,365,123,110
3,sach_se_nhac,1147,383,375
4,sach_se_khen,892,304,310
5,sach_se_che,285,82,76
6,vi_tri_nhac,1046,339,358
7,vi_tri_khen,923,298,315
8,vi_tri_che,163,47,54
9,phong_nhac,1472,511,525


,aspect,polarity,f1,precision,recall,accuracy,kappa,mcc,bal_acc,ho_tro
0,nhan_vien,khen,0.912738,0.906375,0.919192,0.907249,0.813773,0.813864,0.906549,495
1,nhan_vien,che,0.654028,0.683168,0.627273,0.922175,0.610274,0.610980,0.794313,110
2,sach_se,khen,0.971246,0.962025,0.980645,0.980810,0.956848,0.956947,0.980768,310
3,sach_se,che,0.828947,0.828947,0.828947,0.972281,0.813866,0.813866,0.906933,76
4,vi_tri,khen,0.893683,0.868263,0.920635,0.926439,0.837522,0.838361,0.925004,315
5,vi_tri,che,0.568807,0.563636,0.574074,0.949893,0.542211,0.542237,0.773462,54
6,phong,khen,0.767584,0.744807,0.791798,0.837953,0.643376,0.644086,0.826656,317
7,phong,che,0.791128,0.801498,0.781022,0.879531,0.706503,0.706620,0.850601,274
8,gia_tri,khen,0.846442,0.875969,0.818841,0.956290,0.820994,0.821629,0.899420,138
9,gia_tri,che,0.571429,0.651163,0.509091,0.955224,0.548180,0.552805,0.746052,55


Khía cạnh khó nhất theo F1 trung bình khen/chê: tien_nghi
Nhãn khó nhất: {'nhan': 'vi_tri_che', 'f1': 0.5688073394495413}
ABSA-F1: 0.7561255284105146 CI95: [0.7393548036272827, 0.7717808702972868]
Giới hạn: validation dưới 0,80; chưa có điểm private.


## Năm ví dụ lỗi từ validation

Đây là các dòng lấy từ **train đã tách**, không phải test. Các cặp nhãn ghi
đúng nhãn gốc và nhãn mô hình; nhận xét là giải thích khả dĩ, không sửa đáp án.

In [ ]:
comments = {
    'train_0008':'Viết tắt nv/dthuong làm mất lời khen nhân viên; phủ định chưa sạch dễ bị nhận nhầm thành khen sạch. Có từ phòng không đồng nghĩa nhãn phong_nhac: cần theo quy ước khía cạnh.',
    'train_0011':'Mùi hôi nhà vệ sinh thuộc sạch sẽ theo nhãn gốc; mô hình lan sang phòng và nhân viên vì tương quan từ vựng.',
    'train_0012':'Đoạn dài có nhiều khía cạnh, tương phản và bù trừ; mô hình khó gắn riêng lời chê buffet, đường sửa và tiện nghi theo đúng nhãn.',
    'train_0013':'Mùi thơm là tín hiệu gián tiếp về sạch sẽ, dễ bị mô hình TF-IDF bỏ sót.',
    'train_0022':'Cần tốt hơn là lời chê nhẹ về bữa sáng; từ tốt có thể khiến mô hình dự đoán nhầm lời khen.'
}
errors = json.loads((OUT/'validation_errors.json').read_text(encoding='utf-8'))
for example in errors[:5]:
    print('ID:', example['id'])
    print(example['text'])
    print('Sai:', json.dumps(example['errors'], ensure_ascii=False))
    print('Nhận xét:', comments.get(example['id'], 'Cần xem quan hệ giữa khía cạnh, phủ định và ngữ cảnh.'))
    print()

## Kiểm tra định dạng và tái lập từ mô hình đã lưu

Mô hình cuối fit lại trên toàn bộ train, dùng ngưỡng đã chọn trên tuning.
Không biết điểm test. Không điều chỉnh dự đoán theo văn bản hoặc nhãn test.

In [5]:
import hashlib
import joblib
import numpy as np
info = json.loads((OUT/'run_info.json').read_text())
print('Phần cứng và thời gian:', json.dumps(info, ensure_ascii=False, indent=2))
assert score_absa.kiem_tra(OUT/'submission.csv', DATA/'test_blind.csv') == 0
bundle = joblib.load(OUT/'model.joblib')
# Chỉ mô hình cổ điển đọc text ở bước suy luận; không in text.
test = pd.read_csv(DATA/'test_blind.csv')
with threadpool_limits(limits=1):
    scores = bundle['model'].decision_function(bundle['vectorizer'].transform(test.text.map(pipeline.normalize)))
reproduced = pipeline.frame(test.id, pipeline.constrained(scores, bundle['thresholds']))
submitted = pd.read_csv(OUT/'submission.csv')
assert reproduced.equals(submitted)
print('Nạp lại mô hình: toàn bộ dự đoán khớp chính xác.')
print('SHA-256 submission:', hashlib.sha256((OUT/'submission.csv').read_bytes()).hexdigest())

Phần cứng và thời gian: {
  "python": "3.12.14",
  "os": "Linux-6.18.57-x86_64-with-glibc2.41",
  "cpu": "INTEL(R) XEON(R) PLATINUM 8573C",
  "logical_cpu": 3,
  "ram": "MemTotal:       10207480 kB",
  "gpu": "Not used; CPU only",
  "feature_seconds": 4.311222114999964,
  "full_fit_seconds": 12.487642709999932,
  "test_predict_seconds": 0.5743376219999163,
  "total_seconds": 37.01031709900002,
  "selected": "nbsvm",
  "seed": 2026,
  "train_rows": 4688,
  "test_rows": 732,
  "source_commit": "7c052a71010eac9361ea8803b9e8bdd6cfc99be8",
  "sha256": {
    "train.csv": "665b581d11c5cb59eb5b3e6e730660fdd28567c64392f2e288e375dd082bd95e",
    "test_blind.csv": "a8452bbc921f8a3442d318fc2488ae6f4df87d04c160ddcb5f5b3b8d11473162",
    "submission_demo.csv": "4a58d1c42010996fd648317a23c6da9358779b15d25f8b74a901bdb5e7e2a30e",
    "score_absa.py": "423bb1c5be023b8ce04264f5a72bdf01ba6844a7fa39f87267b0bfe5116de333"
  },
  "submission_sha256": "15f913519d5ec57d1de452709c0766278d889337b3d3e7c7c3b3b2ba53

Nạp lại mô hình: toàn bộ dự đoán khớp chính xác.
SHA-256 submission: 15f913519d5ec57d1de452709c0766278d889337b3d3e7c7c3b3b2ba53c17ce2


## Kết luận thực nghiệm

NB-SVM cải thiện baseline trên cùng giao thức nhưng validation vẫn dưới mốc
0,80. Không có căn cứ khẳng định đạt mốc private. Cần thử mô hình ngữ cảnh như
PhoBERT/ViSoBERT, tăng độ phủ từ viết tắt và đánh giá trên holdout mới hoặc nested
CV. Phân bố lời chê của train khác test là một rủi ro; CI bootstrap hiện tại không
phản ánh dịch chuyển đó. Không sinh thêm dữ liệu nên không có thí nghiệm ablation
dữ liệu bổ sung. Không đóng gói train.csv/test_blind.csv gốc.